In [1]:
import sys
import os
sys.path.append(r'C:\Users\emira\OneDrive\Desktop\main projem')
os.chdir(r'C:\Users\emira\OneDrive\Desktop\main projem')

import numpy as np
import pandas as pd
import json
import joblib
from src.feature_extractor import FeaturePipeline

# Joblib ile yükle (pickle değil)
model = joblib.load('data/models/baseline_random_forest.pkl')

with open('data/processed/feature_names.json') as f:
    feature_names = json.load(f)
pipeline = FeaturePipeline()
X_test = np.load('data/processed/X_test.npy')
y_test = np.load('data/processed/y_test.npy')

print("Herşey yüklendi!")
print(f"Model tipi: {type(model)}")

Herşey yüklendi!
Model tipi: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [2]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

mesru_idx = np.where(y_test == 0)[0]
phishing_idx = np.where(y_test == 1)[0]

print(f"Meşru URL'lerin ortalama phishing prob: {y_prob[mesru_idx].mean():.4f}")
print(f"Phishing URL'lerin ortalama phishing prob: {y_prob[phishing_idx].mean():.4f}")
print(f"\nMeşru URL'lerin yanlış tahmin oranı:")
print(f"  prob > 0.5: {(y_prob[mesru_idx] > 0.5).mean()*100:.1f}%")
print(f"  prob > 0.7: {(y_prob[mesru_idx] > 0.7).mean()*100:.1f}%")

Meşru URL'lerin ortalama phishing prob: 0.1255
Phishing URL'lerin ortalama phishing prob: 0.8692

Meşru URL'lerin yanlış tahmin oranı:
  prob > 0.5: 6.7%
  prob > 0.7: 2.2%


In [3]:
import os

# Lexicon dosyasını bul
for root, dirs, files in os.walk('.'):
    for file in files:
        if 'brand' in file.lower() or 'lexicon' in file.lower() or 'turkish' in file.lower():
            print(os.path.join(root, file))

In [4]:
# src klasöründe ne var?
for root, dirs, files in os.walk('src'):
    for file in files:
        print(os.path.join(root, file))

src\adversarial.py
src\cialdini.py
src\data_collector.py
src\defense.py
src\feature_extractor.py
src\feature_extractor_backup.py
src\utils.py
src\__init__.py
src\__pycache__\adversarial.cpython-310.pyc
src\__pycache__\cialdini.cpython-310.pyc
src\__pycache__\data_collector.cpython-310.pyc
src\__pycache__\defense.cpython-310.pyc
src\__pycache__\feature_extractor.cpython-310.pyc
src\__pycache__\utils.cpython-310.pyc
src\__pycache__\__init__.cpython-310.pyc


In [5]:
# feature_extractor.py içinde marka listesi nerede?
with open('src/feature_extractor.py', 'r', encoding='utf-8') as f:
    content = f.read()

# Türkçe marka ile ilgili satırları bul
for i, line in enumerate(content.split('\n')):
    if 'brand' in line.lower() or 'onedio' in line.lower() or 'turkish' in line.lower() or 'marka' in line.lower():
        print(f"Satır {i}: {line}")

Satır 17: KNOWN_BRANDS = [
Satır 18:     # Global markalar
Satır 33:     "rebrand", "short", "buff", "ift", "dlvr",
Satır 35:     "onedio", "hurriyet", "sabah", "milliyet", "cumhuriyet",
Satır 178:             lev_scores = self._levenshtein_all_brands(domain)
Satır 179:             features["min_brand_levenshtein"]    = lev_scores["min_dist"]
Satır 180:             features["min_brand_levenshtein_norm"] = lev_scores["min_norm"]
Satır 181:             features["closest_brand"]            = lev_scores["closest"]
Satır 182:             features["is_near_brand"]            = int(lev_scores["min_dist"] <= 2 and
Satır 184:             features["is_exact_brand"]           = int(lev_scores["min_dist"] == 0)
Satır 187:             brand_in_domain = self._brand_in_domain(domain)
Satır 188:             features["brand_in_non_brand_domain"] = brand_in_domain
Satır 194:                 "min_brand_levenshtein": 999,
Satır 195:                 "min_brand_levenshtein_norm": 1.0,
Satır 196:            

In [6]:
# KNOWN_BRANDS listesini göster
with open('src/feature_extractor.py', 'r', encoding='utf-8') as f:
    lines = f.readlines()

# 17. satırdan itibaren listeyi göster
for i in range(16, 80):
    print(f"{i+1}: {lines[i]}", end='')

17: 
18: KNOWN_BRANDS = [
19:     # Global markalar
20:     "google", "youtube", "facebook", "amazon", "wikipedia", "twitter",
21:     "instagram", "linkedin", "reddit", "netflix", "microsoft", "apple",
22:     "github", "stackoverflow", "paypal", "ebay", "walmart", "dropbox",
23:     "spotify", "twitch", "adobe", "oracle", "ibm", "salesforce", "zoom",
24:     "slack", "trello", "notion", "canva", "cloudflare", "bankofamerica",
25:     "chase", "wellsfargo", "citibank", "hsbc", "barclays", "americanexpress",
26:     "visa", "mastercard", "whatsapp", "telegram", "tiktok", "snapchat",
27:     "yahoo", "bing", "duckduckgo", "pinterest", "tumblr", "quora",
28:     "medium", "wordpress", "blogger", "wix", "squarespace", "shopify",
29:     "stripe", "square", "coinbase", "binance", "kraken", "opensea",
30:     "uber", "lyft", "airbnb", "booking", "tripadvisor", "expedia",
31:     "aliexpress", "alibaba", "etsy", "rakuten", "wish",
32:     # URL kısaltıcılar - MEŞRU
33:     "bitly", "bit", "t

In [7]:
with open('src/feature_extractor.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_brands = '''KNOWN_BRANDS = [
    "google", "youtube", "facebook", "amazon", "wikipedia", "twitter",
    "instagram", "linkedin", "reddit", "netflix", "microsoft", "apple",
    "github", "stackoverflow", "paypal", "ebay", "walmart", "dropbox",
    "spotify", "twitch", "adobe", "oracle", "ibm", "salesforce", "zoom",
    "slack", "trello", "notion", "canva", "cloudflare", "bankofamerica",
    "chase", "wellsfargo", "citibank", "hsbc", "barclays", "americanexpress",
    "visa", "mastercard", "whatsapp", "telegram", "tiktok", "snapchat",
]'''

new_brands = '''KNOWN_BRANDS = [
    # Global markalar
    "google", "youtube", "facebook", "amazon", "wikipedia", "twitter",
    "instagram", "linkedin", "reddit", "netflix", "microsoft", "apple",
    "github", "stackoverflow", "paypal", "ebay", "walmart", "dropbox",
    "spotify", "twitch", "adobe", "oracle", "ibm", "salesforce", "zoom",
    "slack", "trello", "notion", "canva", "cloudflare", "bankofamerica",
    "chase", "wellsfargo", "citibank", "hsbc", "barclays", "americanexpress",
    "visa", "mastercard", "whatsapp", "telegram", "tiktok", "snapchat",
    "yahoo", "bing", "duckduckgo", "pinterest", "tumblr", "quora",
    "medium", "wordpress", "blogger", "wix", "squarespace", "shopify",
    "stripe", "square", "coinbase", "binance", "kraken", "opensea",
    "uber", "lyft", "airbnb", "booking", "tripadvisor", "expedia",
    "aliexpress", "alibaba", "etsy", "rakuten", "wish",
    # URL kısaltıcılar - MEŞRU
    "bitly", "bit", "tinyurl", "tiny", "goo", "ow", "t",
    "rebrand", "short", "buff", "ift", "dlvr",
    # Türkçe haber ve medya
    "onedio", "hurriyet", "sabah", "milliyet", "cumhuriyet",
    "haberturk", "ntv", "cnnturk", "trt", "sozcu",
    "mynet", "ensonhaber", "takvim", "posta", "aksam",
    "bloomberght", "dunya", "ekonomist", "forbes",
    # Türkçe e-ticaret
    "trendyol", "hepsiburada", "gittigidiyor", "pttavm",
    "sahibinden", "arabam", "ikinciyeni", "dolap",
    "yemeksepeti", "getir", "trendyolmarket", "migros",
    "amazon", "morhipo", "boyner", "lcwaikiki", "defacto",
    # Türkçe bankalar
    "akbank", "garanti", "isbank", "ziraatbank", "vakifbank",
    "enpara", "finansbank", "qnb", "denizbank", "yapikredi",
    "halkbank", "teb", "ing", "odea", "albaraka", "kuveytturk",
    # Türkçe telekom
    "turkcell", "vodafone", "turktelekom", "turk", "turknet",
    "superonline", "millenicom", "kablonet",
    # Türkçe devlet
    "gov", "turkiye", "egov", "edevlet", "meb", "saglik",
    "tubitak", "btk", "spk", "bddk", "tcmb",
    # Türkçe diğer
    "eksi", "uludagsozluk", "instela", "twitter",
    "biletix", "passo", "ticketmaster",
]'''

content = content.replace(old_brands, new_brands)

with open('src/feature_extractor.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("Lexicon güncellendi!")

Lexicon güncellendi!


In [8]:
# Kernel restart gerek, cache temizleme
import importlib
import src.feature_extractor
importlib.reload(src.feature_extractor)
from src.feature_extractor import FeaturePipeline
pipeline = FeaturePipeline()

test_urls = [
    "https://google.com",
    "https://gov.tr",
    "https://wikipedia.org",
    "https://sahibinden.com",
    "https://bit.ly/3abc123",
    "https://tinyurl.com/xyz",
    "https://onedio.com",
    "http://paypal-login-verify.xyz",
    "http://amazon-secure-update.tk",
]

print("URL Analizi:")
print("-" * 60)
for url in test_urls:
    df = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = model.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.5 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

URL Analizi:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://google.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://gov.tr

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://wikipedia.org

 Feature extraction basliyor...
   [1/4] 

In [9]:
# Threshold değiştir ve tekrar test et
THRESHOLD = 0.7

print("Threshold 0.7 ile test:")
print("-" * 60)
for url in test_urls:
    df = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = model.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > THRESHOLD else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Threshold 0.7 ile test:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://google.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://gov.tr

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://wikipedia.org

 Feature extraction basliyor..

In [10]:
THRESHOLD = 0.72

print("Threshold 0.72 ile test:")
print("-" * 60)
for url in test_urls:
    df = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = model.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > THRESHOLD else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Threshold 0.72 ile test:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://google.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://gov.tr

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://wikipedia.org

 Feature extraction basliyor.

In [11]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

for i, line in enumerate(content.split('\n')):
    if 'predict' in line.lower() and 'model' in line.lower():
        print(f"Satır {i}: {line}")

Satır 292:                 prob = model.predict_proba(X)[0][1]
Satır 419:                     prob_adv = model.predict_proba(X_adv)[0][1]
Satır 421:                     prob_def = defended_model.predict_proba(X_adv)[0][1]


In [12]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

# Threshold 0.5 olan predict satırlarını 0.72 ile değiştir
old1 = "prediction = model.predict(X)[0]"
new1 = "prediction = 1 if prob > 0.72 else 0"

old2 = "pred_adv = model.predict(X_adv)[0]"
new2 = "pred_adv = 1 if prob_adv > 0.72 else 0"

old3 = "pred_def = defended_model.predict(X_adv)[0]"
new3 = "pred_def = 1 if prob_def > 0.72 else 0"

content = content.replace(old1, new1)
content = content.replace(old2, new2)
content = content.replace(old3, new3)

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("Threshold 0.72 app.py'ye yansıtıldı!")

Threshold 0.72 app.py'ye yansıtıldı!


In [13]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

# Değişiklik yapıldı mı kontrol et
for i, line in enumerate(content.split('\n')):
    if 'prediction' in line and ('0.72' in line or 'predict' in line.lower()):
        print(f"Satır {i}: {line}")

Satır 293:                 prediction = 1 if prob > 0.50 else 0
Satır 301:                     if prediction == 1:
Satır 329:                         <div class="info-row"><span class="info-key">VERDICT</span><span class="info-val" style="color:{'#ff006e' if prediction==1 else '#00ff41'};">{'MALICIOUS' if prediction==1 else 'LEGITIMATE'}</span></div>
Satır 368:                 if prediction == 1 and cld_result["total_score"] > 0.1:
Satır 405:                 if prediction == 1:


In [14]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

for i, line in enumerate(content.split('\n')):
    if 'load_model' in line or 'model_name' in line.lower():
        print(f"Satır {i}: {line}")

Satır 12: from src.utils import load_model
Satır 263:     defended_model = load_model('defended_random_forest_robust')


In [15]:
for i, line in enumerate(content.split('\n')):
    if i in range(260, 270):
        print(f"Satır {i}: {line}")

Satır 260: def load_resources():
Satır 261:     import joblib
Satır 262:     model = joblib.load('data/models/baseline_random_forest.pkl')
Satır 263:     defended_model = load_model('defended_random_forest_robust')
Satır 264:     with open('data/processed/feature_names.json') as f:
Satır 265:         feature_names = json.load(f)
Satır 266:     pipeline = FeaturePipeline()
Satır 267:     cialdini = CialdiniAnalyzer()
Satır 268:     return model, defended_model, feature_names, pipeline, cialdini
Satır 269: 


In [16]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

old = "model = joblib.load('models/best_xgboost.pkl')"
new = "model = load_model('baseline_random_forest')"

content = content.replace(old, new)

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("Model değiştirildi! Random Forest kullanılacak.")

Model değiştirildi! Random Forest kullanılacak.


In [17]:
import os
for root, dirs, files in os.walk('data'):
    for file in files:
        print(os.path.join(root, file))

data\models\baseline_logistic_regression.pkl
data\models\baseline_random_forest.pkl
data\models\baseline_xgboost.pkl
data\models\defended_ensemble.pkl
data\models\defended_logistic_regression_robust.pkl
data\models\defended_random_forest_robust.pkl
data\models\defended_xgboost_robust.pkl
data\processed\attack_results.json
data\processed\feature_matrix.csv
data\processed\feature_names.json
data\processed\final_summary.csv
data\processed\raw_dataset.csv
data\processed\robustness_ranking.csv
data\processed\X_adv_combined.npy
data\processed\X_adv_gaussian.npy
data\processed\X_adv_minimal.npy
data\processed\X_test.npy
data\processed\X_train.npy
data\processed\y_test.npy
data\processed\y_train.npy
data\raw\kaggle_phishing.csv
data\raw\phishing_site_urls.csv
data\raw\raw_dataset.csv
data\raw\urldata.csv


In [18]:
import pandas as pd

df = pd.read_csv('data/processed/raw_dataset.csv')
print(f"Toplam satır: {len(df)}")
print(f"Kolonlar: {df.columns.tolist()}")
print(f"\nLabel dağılımı:")
print(df['label'].value_counts())
print(f"\nİlk 5 satır:")
print(df.head())

Toplam satır: 50000
Kolonlar: ['url', 'timestamp', 'label', 'source']

Label dağılımı:
label
0    25000
1    25000
Name: count, dtype: int64

İlk 5 satır:
                                                 url  \
0                      kansascity-lifeinsurance.org/   
1  http://www.tokabrasil.com.br/~wellsfargo0o/cgi...   
2                                tinyurl.com/ju7vx9b   
3  https://www.kunden-kundenservices.com/vetos/an...   
4                    youtube.com/watch?v=wkbRmmVGRjk   

                   timestamp  label  source  
0  2024-08-02 00:00:00+00:00      0  kaggle  
1  2024-06-14 00:00:00+00:00      1  kaggle  
2  2023-03-26 00:00:00+00:00      1  kaggle  
3  2023-09-22 00:00:00+00:00      1  kaggle  
4  2024-01-20 00:00:00+00:00      0  kaggle  


In [19]:
# Meşru URL'lerin örnekleri
mesru = df[df['label'] == 0]['url'].sample(20, random_state=42).tolist()
print("Mevcut meşru URL örnekleri:")
for url in mesru:
    print(f"  {url}")

Mevcut meşru URL örnekleri:
  enotes.com/topic/Armenians
  tools.ietf.org/html/rfc3115
  checkoutmycards.com/Players/Baseball/V
  nosorigines.qc.ca/genealogielistfirstname.aspx?Family=Leguerrier_8537&lng=en
  https://www.stjameskemptville.com/
  mccord-museum.qc.ca/scripts/printtour.php?tourID=VQ_P4_3_EN&Lang=2
  tvguide.com/celebrities/victor-mature/178273
  https://www.tlavideo.com/gay-ty-lattimore/person-53614-3
  carmelunited.org/
  https://www.mapsof.net/bamako
  https://www.newburyportnews.com/sports/x845817669/Repeaters-Plouffe-Flinn-earn-5K-titles-for-the-second-year-in-a-row/print
  en.wikipedia.org/wiki/Hal_Jordan
  https://www.onlineradio2.com/listen/CKOI_1021
  https://www.youtube.com/watch?v=PZqxbGnEF24
  flickr.com/photos/foxypar4/3222324382/
  radioalice.radio.com/
  https://www.britannica.com
  sites.google.com/site/agrrho/
  therepublikofmancunia.com/video-liverpool-2-3-united-fa-youth-cup-quarter-finals/
  https://www.tvguide.com/celebrities/maurice-benard/256941


In [20]:
import requests
import zipfile
import io

print("Tranco listesi indiriliyor...")
try:
    r = requests.get("https://tranco-list.eu/top-1m.csv.zip", timeout=30)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    tranco = pd.read_csv(z.open(z.namelist()[0]), header=None, names=['rank', 'domain'])
    print(f"Tranco listesi indirildi: {len(tranco)} site")
    print(tranco.head(10))
except Exception as e:
    print(f"Hata: {e}")

Tranco listesi indiriliyor...
Tranco listesi indirildi: 1000000 site
   rank            domain
0     1        google.com
1     2  gtld-servers.net
2     3       gstatic.com
3     4      facebook.com
4     5    cloudflare.com
5     6     microsoft.com
6     7    googleapis.com
7     8           mail.ru
8     9       youtube.com
9    10         apple.com


In [21]:
import numpy as np
from datetime import datetime

# Top 10000 siteyi al (ilk 10K en güvenilir)
top_sites = tranco[tranco['rank'] <= 10000]['domain'].tolist()

# Türkçe siteleri de ekle
turkish_sites = [
    "google.com.tr", "gov.tr", "edu.tr", "onedio.com",
    "sahibinden.com", "hepsiburada.com", "trendyol.com",
    "yemeksepeti.com", "akbank.com", "garantibbva.com",
    "isbank.com.tr", "ziraatbank.com", "vakifbank.com.tr",
    "turkcell.com.tr", "vodafone.com.tr", "ntv.com.tr",
    "hurriyet.com.tr", "sabah.com.tr", "milliyet.com.tr",
    "cumhuriyet.com.tr", "haberturk.com", "sozcu.com.tr",
    "trt.net.tr", "pttavm.com", "gittigidiyor.com",
    "getir.com", "eksi.sozluk.com", "mynet.com",
]

# URL formatına çevir
new_urls = [f"https://{domain}" for domain in top_sites[:5000]]
new_urls += [f"https://{domain}" for domain in turkish_sites]

print(f"Eklenecek meşru URL sayısı: {len(new_urls)}")
print("İlk 5 örnek:")
for url in new_urls[:5]:
    print(f"  {url}")

Eklenecek meşru URL sayısı: 5028
İlk 5 örnek:
  https://google.com
  https://gtld-servers.net
  https://gstatic.com
  https://facebook.com
  https://cloudflare.com


In [22]:
# Yeni meşru URL'leri dataframe'e ekle
new_df = pd.DataFrame({
    'url': new_urls,
    'timestamp': datetime.now(),
    'label': 0,
    'source': 'tranco'
})

# Mevcut dataset ile birleştir
df_original = pd.read_csv('data/processed/raw_dataset.csv')
df_combined = pd.concat([df_original, new_df], ignore_index=True)

print(f"Orijinal dataset: {len(df_original)} satır")
print(f"Yeni dataset: {len(df_combined)} satır")
print(f"\nLabel dağılımı:")
print(df_combined['label'].value_counts())

Orijinal dataset: 50000 satır
Yeni dataset: 55028 satır

Label dağılımı:
label
0    30028
1    25000
Name: count, dtype: int64


In [23]:
# Sadece yeni URL'lerin feature'larını çıkar
from src.feature_extractor import FeaturePipeline
import json

pipeline = FeaturePipeline()

print("Yeni URL'lerin feature'ları çıkarılıyor...")
print("Bu 10-15 dakika sürebilir, bekle...")

new_features = pipeline.transform(new_df, verbose=True)

print(f"\nTamamlandı! Shape: {new_features.shape}")

Yeni URL'lerin feature'ları çıkarılıyor...
Bu 10-15 dakika sürebilir, bekle...

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...


URL Features: 100%|██████████████████████████████████████████████████████████████████████| 5028/5028 [00:00<00:00, 26956.05it/s]


   [2/4] Adversarial features cikariliyor...


Adversarial Features: 100%|████████████████████████████████████████████████████████████████| 5028/5028 [00:11<00:00, 424.82it/s]


   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...


Cialdini Features: 100%|█████████████████████████████████████████████████████████████████| 5028/5028 [00:00<00:00, 39219.88it/s]

Feature extraction tamamlandi: 5028 satir, 64 feature

Tamamlandı! Shape: (5028, 67)


In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib

# Mevcut train/test verisini yükle
X_train_old = np.load('data/processed/X_train.npy')
X_test_old = np.load('data/processed/X_test.npy')
y_train_old = np.load('data/processed/y_train.npy')
y_test_old = np.load('data/processed/y_test.npy')

# Yeni feature'ları hazırla
feature_cols = [c for c in new_features.columns if c not in ['url', 'label', 'timestamp', 'source']]
X_new = new_features[feature_cols].reindex(columns=feature_names, fill_value=0).values
y_new = np.zeros(len(X_new))  # hepsi meşru (label=0)

# Birleştir
X_train_combined = np.vstack([X_train_old, X_new])
y_train_combined = np.concatenate([y_train_old, y_new])

print(f"Eski train: {X_train_old.shape}")
print(f"Yeni train: {X_train_combined.shape}")

# Modeli eğit
print("\nModel eğitiliyor...")
rf_new = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_new.fit(X_train_combined, y_train_combined)

# Test et
y_pred = rf_new.predict(X_test_old)
y_prob = rf_new.predict_proba(X_test_old)[:,1]

acc = accuracy_score(y_test_old, y_pred)
f1 = f1_score(y_test_old, y_pred)
auc = roc_auc_score(y_test_old, y_prob)

print(f"\nYeni Model Sonuçları:")
print(f"Accuracy: {acc:.4f}")
print(f"F1: {f1:.4f}")
print(f"AUC: {auc:.4f}")

Eski train: (24987, 64)
Yeni train: (30015, 64)

Model eğitiliyor...

Yeni Model Sonuçları:
Accuracy: 0.9349
F1: 0.9353
AUC: 0.9840


In [25]:
# Yeni modelle test et
test_urls = [
    "https://google.com",
    "https://gov.tr",
    "https://wikipedia.org",
    "https://onedio.com",
    "https://sahibinden.com",
    "https://bit.ly/3abc123",
    "http://paypal-login-verify.xyz",
    "http://amazon-secure-update.tk",
]

print("Yeni model ile test (threshold 0.72):")
print("-" * 60)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = rf_new.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.72 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Yeni model ile test (threshold 0.72):
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://google.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://gov.tr

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://wikipedia.org

 Feature extract

In [26]:
print("Yeni model ile test (threshold 0.50):")
print("-" * 60)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = rf_new.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Yeni model ile test (threshold 0.50):
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://google.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://gov.tr

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://wikipedia.org

 Feature extract

In [27]:
# Hiç train setinde olmayan meşru siteler
hic_gorulmemis = [
    "https://bbc.com",
    "https://cnn.com", 
    "https://nytimes.com",
    "https://eksi.sozluk.gen.tr",
    "https://tr.wikipedia.org",
    "https://internethaber.com",
]

print("Hiç görülmemiş meşru siteler:")
print("-" * 60)
for url in hic_gorulmemis:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = rf_new.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Hiç görülmemiş meşru siteler:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://bbc.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://cnn.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://nytimes.com

 Feature extraction basliyor

In [28]:
import joblib

# Yeni modeli kaydet (eskinin üzerine yaz)
joblib.dump(rf_new, 'data/models/baseline_random_forest.pkl')
print("✅ Yeni model kaydedildi!")

# App.py'deki threshold'u da 0.50'ye düzelt
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

content = content.replace(
    "prediction = 1 if prob > 0.72 else 0",
    "prediction = 1 if prob > 0.50 else 0"
)
content = content.replace(
    "pred_adv = 1 if prob_adv > 0.72 else 0",
    "pred_adv = 1 if prob_adv > 0.50 else 0"
)
content = content.replace(
    "pred_def = 1 if prob_def > 0.72 else 0",
    "pred_def = 1 if prob_def > 0.50 else 0"
)

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("✅ App.py threshold 0.50'ye güncellendi!")

✅ Yeni model kaydedildi!
✅ App.py threshold 0.50'ye güncellendi!


In [29]:
import joblib

# joblib ile kaydet
joblib.dump(rf_new, 'data/models/baseline_random_forest.pkl')

# Bir de models/ klasörüne kaydet (app.py oraya bakıyor olabilir)
import os
os.makedirs('models', exist_ok=True)
joblib.dump(rf_new, 'models/baseline_random_forest.pkl')

print("✅ Model iki yere de kaydedildi!")

✅ Model iki yere de kaydedildi!


In [30]:
import joblib
import os

print("Kaydediliyor...")
joblib.dump(rf_new, 'data/models/baseline_random_forest.pkl')
print("1. kayıt tamam")

os.makedirs('models', exist_ok=True)
joblib.dump(rf_new, 'models/baseline_random_forest.pkl')
print("2. kayıt tamam")

print("✅ Bitti!")

Kaydediliyor...
1. kayıt tamam
2. kayıt tamam
✅ Bitti!


In [31]:
print(type(rf_new))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [32]:
import sys
import os
sys.path.append(r'C:\Users\emira\OneDrive\Desktop\main projem')
os.chdir(r'C:\Users\emira\OneDrive\Desktop\main projem')

import numpy as np
import pandas as pd
import json
import joblib
import requests
import zipfile
import io
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from src.feature_extractor import FeaturePipeline
from src.utils import load_model

print("1. Kütüphaneler yüklendi")

# Modeller ve veriler
with open('data/processed/feature_names.json') as f:
    feature_names = json.load(f)
pipeline = FeaturePipeline()
X_train_old = np.load('data/processed/X_train.npy')
X_test = np.load('data/processed/X_test.npy')
y_train_old = np.load('data/processed/y_train.npy')
y_test = np.load('data/processed/y_test.npy')

print("2. Veriler yüklendi")

# Tranco listesi
r = requests.get("https://tranco-list.eu/top-1m.csv.zip", timeout=30)
z = zipfile.ZipFile(io.BytesIO(r.content))
tranco = pd.read_csv(z.open(z.namelist()[0]), header=None, names=['rank', 'domain'])
top_sites = tranco[tranco['rank'] <= 5000]['domain'].tolist()

turkish_sites = [
    "google.com.tr", "gov.tr", "onedio.com", "sahibinden.com",
    "hepsiburada.com", "trendyol.com", "yemeksepeti.com",
    "akbank.com", "garantibbva.com", "turkcell.com.tr",
    "hurriyet.com.tr", "sabah.com.tr", "ntv.com.tr",
]

new_urls = [f"https://{d}" for d in top_sites] + [f"https://{d}" for d in turkish_sites]
new_df = pd.DataFrame({'url': new_urls, 'timestamp': datetime.now(), 'label': 0, 'source': 'tranco'})

print(f"3. {len(new_urls)} yeni URL hazırlandı")

# Feature extraction
new_features = pipeline.transform(new_df, verbose=True)
feature_cols = [c for c in new_features.columns if c not in ['url', 'label', 'timestamp', 'source']]
X_new = new_features[feature_cols].reindex(columns=feature_names, fill_value=0).values
y_new = np.zeros(len(X_new))

print("4. Feature extraction tamamlandı")

# Birleştir ve eğit
X_train_combined = np.vstack([X_train_old, X_new])
y_train_combined = np.concatenate([y_train_old, y_new])

rf_new = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_new.fit(X_train_combined, y_train_combined)

print("5. Model eğitildi")

# Kaydet
os.makedirs('models', exist_ok=True)
joblib.dump(rf_new, 'data/models/baseline_random_forest.pkl')
joblib.dump(rf_new, 'models/baseline_random_forest.pkl')

print("6. Model kaydedildi!")

# Test
y_pred = rf_new.predict(X_test)
y_prob = rf_new.predict_proba(X_test)[:,1]
acc = accuracy_score(y_test, y_pred)
print(f"7. Accuracy: {acc:.4f}")

1. Kütüphaneler yüklendi
2. Veriler yüklendi
3. 5013 yeni URL hazırlandı

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...


URL Features: 100%|██████████████████████████████████████████████████████████████████████| 5013/5013 [00:00<00:00, 27708.51it/s]


   [2/4] Adversarial features cikariliyor...


Adversarial Features: 100%|████████████████████████████████████████████████████████████████| 5013/5013 [00:11<00:00, 423.46it/s]


   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...


Cialdini Features: 100%|█████████████████████████████████████████████████████████████████| 5013/5013 [00:00<00:00, 44942.91it/s]


Feature extraction tamamlandi: 5013 satir, 64 feature
4. Feature extraction tamamlandı
5. Model eğitildi
6. Model kaydedildi!
7. Accuracy: 0.9356


In [33]:
# utils.py'yi düzelt
with open('src/utils.py', 'r', encoding='utf-8') as f:
    content = f.read()

# pickle yerine joblib kullan
old = "import pickle"
new = "import joblib"

old2 = "model = pickle.load(f)"
new2 = "model = joblib.load(path)"

old3 = '''with open(path, "rb") as f:
        model = pickle.load(f)'''
new3 = "model = joblib.load(path)"

content = content.replace(old, new)
content = content.replace(old3, new3)

with open('src/utils.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("utils.py güncellendi!")

utils.py güncellendi!


In [34]:
import sys
import os
sys.path.append(r'C:\Users\emira\OneDrive\Desktop\main projem')
os.chdir(r'C:\Users\emira\OneDrive\Desktop\main projem')

# utils.py'yi düzelt
with open('src/utils.py', 'r', encoding='utf-8') as f:
    content = f.read()

print(content[2800:3200])  # load_model fonksiyonunu göster

path, "wb") as f:
        pickle.dump(model, f)
    print(f"Model kaydedildi: {path}")


def load_model(name: str):
    """Kaydedilmiş modeli yükler."""
    path = DATA_MODELS / f"{name}.pkl"
    model = joblib.load(path)
    logger.info(f"Model yüklendi: {path}")
    return model


def save_json(data: dict, filename: str, folder=DATA_PROCESSED):
    """Dict'i JSON olarak kaydeder."""
    path = P


In [35]:
print("merhaba")

merhaba


In [36]:
import os
import sys
sys.path.append(r'C:\Users\emira\OneDrive\Desktop\main projem')
os.chdir(r'C:\Users\emira\OneDrive\Desktop\main projem')

# utils.py'deki load_model fonksiyonunu düzelt
with open('src/utils.py', 'r', encoding='utf-8') as f:
    content = f.read()

# pickle yerine joblib kullan
content = content.replace(
    'with open(path, "rb") as f:\n        model = pickle.load(f)',
    'import joblib\n    model = joblib.load(path)'
)

with open('src/utils.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("utils.py düzeltildi!")

utils.py düzeltildi!


In [37]:
test_urls = [
    "https://google.com",
    "https://gov.tr",
    "https://onedio.com",
    "https://sahibinden.com",
    "http://paypal-login-verify.xyz",
    "http://amazon-secure-update.tk",
]

print("Test sonuçları:")
print("-" * 60)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = model.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Test sonuçları:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://google.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://gov.tr

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://onedio.com

 Feature extraction basliyor...
   [1/4] 

In [38]:
import joblib

# Kaydedilen modeli geri yükle ve test et
model_loaded = joblib.load('data/models/baseline_random_forest.pkl')

url = "https://google.com"
df_test = pd.DataFrame({'url': [url], 'label': [0]})
features = pipeline.transform(df_test, verbose=False)
feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
prob = model_loaded.predict_proba(X)[0][1]
print(f"Kaydedilen model → google.com: {prob:.4f}")
print(f"rf_new → google.com: {model.predict_proba(X)[0][1]:.4f}")


 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
Kaydedilen model → google.com: 0.0000
rf_new → google.com: 0.0000


In [39]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

for i, line in enumerate(content.split('\n')):
    if i in range(258, 272):
        print(f"Satır {i}: {line}")

Satır 258: 
Satır 259: # ── LOAD RESOURCES ──
Satır 260: def load_resources():
Satır 261:     import joblib
Satır 262:     model = joblib.load('data/models/baseline_random_forest.pkl')
Satır 263:     defended_model = load_model('defended_random_forest_robust')
Satır 264:     with open('data/processed/feature_names.json') as f:
Satır 265:         feature_names = json.load(f)
Satır 266:     pipeline = FeaturePipeline()
Satır 267:     cialdini = CialdiniAnalyzer()
Satır 268:     return model, defended_model, feature_names, pipeline, cialdini
Satır 269: 
Satır 270: model, defended_model, feature_names, pipeline, cialdini = load_resources()
Satır 271: 


In [40]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

content = content.replace(
    '@st.cache_resource\ndef load_resources():',
    '@st.cache_resource(ttl=0)\ndef load_resources():'
)

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("Cache sıfırlandı!")

Cache sıfırlandı!


In [41]:
with open('src/utils.py', 'r', encoding='utf-8') as f:
    lines = f.readlines()

for i in range(100, 120):
    print(f"{i}: {lines[i]}", end='')
    

100:         pickle.dump(model, f)
101:     print(f"Model kaydedildi: {path}")
102: 
103: 
104: def load_model(name: str):
105:     """Kaydedilmiş modeli yükler."""
106:     path = DATA_MODELS / f"{name}.pkl"
107:     model = joblib.load(path)
108:     logger.info(f"Model yüklendi: {path}")
109:     return model
110: 
111: 
112: def save_json(data: dict, filename: str, folder=DATA_PROCESSED):
113:     """Dict'i JSON olarak kaydeder."""
114:     path = Path(folder) / f"{filename}.json"
115:     with open(path, "w", encoding="utf-8") as f:
116:         json.dump(data, f, indent=2, ensure_ascii=False, default=str)
117:     logger.info(f"JSON kaydedildi: {path}")
118: 
119: 


In [42]:
# app.py'de load_model yerine direkt joblib kullanalım
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

old = "    model = load_model('baseline_random_forest')"
new = "    import joblib\n    model = joblib.load('data/models/baseline_random_forest.pkl')"

content = content.replace(old, new)

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("app.py güncellendi!")

app.py güncellendi!


In [43]:
# Cache'i tamamen devre dışı bırak
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

content = content.replace(
    '@st.cache_resource(ttl=0)\ndef load_resources():',
    'def load_resources():'
)
content = content.replace(
    '@st.cache_resource\ndef load_resources():',
    'def load_resources():'
)

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("Cache tamamen kaldırıldı!")

Cache tamamen kaldırıldı!


In [44]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

for i, line in enumerate(content.split('\n')):
    if 'joblib' in line or 'load_model' in line or 'baseline' in line:
        print(f"Satır {i}: {line}")

Satır 1: import joblib
Satır 12: from src.utils import load_model
Satır 261:     import joblib
Satır 262:     model = joblib.load('data/models/baseline_random_forest.pkl')
Satır 263:     defended_model = load_model('defended_random_forest_robust')


In [45]:
import joblib
from sklearn.ensemble import RandomForestClassifier

# Dosyayı aç
model_check = joblib.load('data/models/baseline_random_forest.pkl')
print(f"Model tipi: {type(model_check)}")
print(f"N estimators: {model_check.n_estimators}")
print(f"N features: {model_check.n_features_in_}")

# google.com test
import pandas as pd
url = "https://google.com"
df_test = pd.DataFrame({'url': [url], 'label': [0]})
features = pipeline.transform(df_test, verbose=False)
feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
prob = model_check.predict_proba(X)[0][1]
print(f"\nDosyadan yüklenen model → google.com: {prob:.4f}")

Model tipi: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
N estimators: 300
N features: 64

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature

Dosyadan yüklenen model → google.com: 0.0000


In [46]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

# 258-270 arası satırları göster
for i, line in enumerate(content.split('\n')):
    if i in range(258, 275):
        print(f"{i}: {line}")

258: 
259: # ── LOAD RESOURCES ──
260: def load_resources():
261:     import joblib
262:     model = joblib.load('data/models/baseline_random_forest.pkl')
263:     defended_model = load_model('defended_random_forest_robust')
264:     with open('data/processed/feature_names.json') as f:
265:         feature_names = json.load(f)
266:     pipeline = FeaturePipeline()
267:     cialdini = CialdiniAnalyzer()
268:     return model, defended_model, feature_names, pipeline, cialdini
269: 
270: model, defended_model, feature_names, pipeline, cialdini = load_resources()
271: 
272: def extract_features(url):
273:     print(f'DEBUG: {url}')
274:     df = pd.DataFrame({'url': [url], 'label': [0]})


In [47]:
# app.py'de tüm model yükleme satırlarını bul
for i, line in enumerate(content.split('\n')):
    if 'xgboost' in line.lower() or 'best_xgb' in line.lower() or 'pkl' in line.lower():
        print(f"Satır {i}: {line}")

Satır 262:     model = joblib.load('data/models/baseline_random_forest.pkl')
Satır 326:                         <div class="info-row"><span class="info-key">MODEL</span><span class="info-val">XGBoost // Zero-Day Split // ACC=94.08%</span></div>
Satır 456:                         <div class="perf-lbl">XGBoost Baseline</div>
Satır 461:                         <div class="perf-lbl">XGBoost + Cialdini Features</div>
Satır 474:                         "Model": ["Classic Only (56 features)", "Cialdini Only (8 features)", "Classic + Cialdini (64)", "XGBoost Baseline", "XGBoost + Cialdini", "RF Robust (Under Attack)"],


In [48]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

old = "def extract_features(url):\n    df = pd.DataFrame({'url': [url], 'label': [0]})"
new = "def extract_features(url):\n    print(f'DEBUG: {url}')\n    df = pd.DataFrame({'url': [url], 'label': [0]})"

content = content.replace(old, new)

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("Debug eklendi!")

Debug eklendi!


In [49]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

old = """def extract_features(url):
    if not url.startswith('http'):
        url = 'https://' + url
    df = pd.DataFrame({'url': [url], 'label': [0]})"""

new = """def extract_features(url):
    # Sadece http veya https yoksa ekle, mevcut protokolü değiştirme
    if not url.startswith('http://') and not url.startswith('https://'):
        url = 'https://' + url
    df = pd.DataFrame({'url': [url], 'label': [0]})"""

content = content.replace(old, new)

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("Güncellendi!")

Güncellendi!


In [50]:
with open('app.py', 'r', encoding='utf-8') as f:
    content = f.read()

# extract_features fonksiyonunu bul ve göster
idx = content.find('def extract_features')
print(content[idx:idx+300])

def extract_features(url):
    print(f'DEBUG: {url}')
    df = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=fea


In [51]:
import os
for root, dirs, files in os.walk(r'C:\Users\emira\OneDrive\Desktop'):
    for file in files:
        if file == 'app.py':
            print(os.path.join(root, file))

C:\Users\emira\OneDrive\Desktop\main projem\app.py


In [52]:
df_original = pd.read_csv('data/processed/raw_dataset.csv')

# Meşru URL'lerin https/http dağılımı
mesru = df_original[df_original['label'] == 0]['url']

https_count = mesru.str.startswith('https://').sum()
http_count = mesru.str.startswith('http://').sum()
no_protocol = (~mesru.str.startswith('http')).sum()

print(f"https:// ile başlayan: {https_count}")
print(f"http:// ile başlayan: {http_count}")
print(f"Protokolsüz: {no_protocol}")
print(f"\nÖrnekler:")
print(mesru.sample(10).tolist())

https:// ile başlayan: 11634
http:// ile başlayan: 104
Protokolsüz: 13260

Örnekler:
['https://www.amazon.com/Pierre-Boileau/e/B0034OQG2U', 'nymag.com/listings/bar/nowhere/', 'https://www.liberta.co.za/', 'www.klariti.com/information-architecture/', 'kids-crafts.blogspot.com/2007/04/cool-paper-airplane-facts.html', 'https://www.tripadvisor.com', 'iskcon-london.org/', 'craneindia.net/', 'https://www.vancouversun.com/sports/Retirees+return+fill+Lions+roster/5608465/story.html', 'https://www.musicstack.com/']


In [53]:
# Protokolsüz URL test et
test_urls = [
    "google.com",
    "wikipedia.org", 
    "github.com",
    "onedio.com",
]

print("Protokolsüz test:")
print("-" * 50)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = model.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Protokolsüz test:
--------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
🔴 PHİSHİNG (0.51) → google.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
🔴 PHİSHİNG (0.54) → wikipedia.org

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.48) → github.com

 Feature extraction basliyor...
   [1/4] URL features cikari

In [54]:
# Train setinde google.com var mı?
df_original = pd.read_csv('data/processed/raw_dataset.csv')
mesru = df_original[df_original['label'] == 0]['url']

google_urls = mesru[mesru.str.contains('google.com', na=False)]
print(f"google.com içeren URL sayısı: {len(google_urls)}")
print(google_urls.head(10).tolist())

google.com içeren URL sayısı: 58
['code.google.com/p/foxreplace/', 'https://www.sketchup.google.com/3dwarehouse/details?mid=69c621d766d2ba8faba853580c1b134c', 'https://www.google.com/finance?cid=690815', 'developers.google.com/blogger/', 'https://www.google.com/finance?cid=662190', 'https://www.plus.google.com/116190287417290527431', 'https://www.sites.google.com/site/cedarsrevolution/', 'https://www.groups.google.com/group/android-porting/browse_thread/thread/dabf53d1ff4c1af5', 'google.com/finance?cid=677349', 'https://www.google.com/finance?cid=21553']


In [55]:
import requests
import zipfile
import io
from datetime import datetime

# Tranco listesi indir
print("Tranco indiriliyor...")
r = requests.get("https://tranco-list.eu/top-1m.csv.zip", timeout=30)
z = zipfile.ZipFile(io.BytesIO(r.content))
tranco = pd.read_csv(z.open(z.namelist()[0]), header=None, names=['rank', 'domain'])
top_sites = tranco[tranco['rank'] <= 5000]['domain'].tolist()

# Her domain için 3 format
new_urls = []
for domain in top_sites:
    new_urls.append(f"https://{domain}")
    new_urls.append(f"http://{domain}")
    new_urls.append(domain)

# Türkçe siteler
turkish_sites = [
    "google.com.tr", "gov.tr", "onedio.com", "sahibinden.com",
    "hepsiburada.com", "trendyol.com", "akbank.com",
    "turkcell.com.tr", "hurriyet.com.tr", "ntv.com.tr",
]
for domain in turkish_sites:
    new_urls.append(f"https://{domain}")
    new_urls.append(f"http://{domain}")
    new_urls.append(domain)

new_df = pd.DataFrame({
    'url': new_urls,
    'timestamp': datetime.now(),
    'label': 0,
    'source': 'tranco'
})

print(f"Toplam eklenecek URL: {len(new_urls)}")

Tranco indiriliyor...
Toplam eklenecek URL: 15030


In [56]:
print("Feature extraction başlıyor, 15-20 dakika sürebilir...")
new_features = pipeline.transform(new_df, verbose=True)
print(f"Tamamlandı! Shape: {new_features.shape}")

Feature extraction başlıyor, 15-20 dakika sürebilir...

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...


URL Features: 100%|████████████████████████████████████████████████████████████████████| 15030/15030 [00:00<00:00, 25784.44it/s]


   [2/4] Adversarial features cikariliyor...


Adversarial Features: 100%|██████████████████████████████████████████████████████████████| 15030/15030 [00:36<00:00, 414.14it/s]


   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...


Cialdini Features: 100%|███████████████████████████████████████████████████████████████| 15030/15030 [00:00<00:00, 28495.25it/s]

Feature extraction tamamlandi: 15030 satir, 64 feature
Tamamlandı! Shape: (15030, 67)


In [57]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib

# Feature'ları hazırla
feature_cols = [c for c in new_features.columns if c not in ['url', 'label', 'timestamp', 'source']]
X_new = new_features[feature_cols].reindex(columns=feature_names, fill_value=0).values
y_new = np.zeros(len(X_new))

# Orijinal train seti ile birleştir
X_train_original = np.load('data/processed/X_train.npy')
y_train_original = np.load('data/processed/y_train.npy')

X_train_combined = np.vstack([X_train_original, X_new])
y_train_combined = np.concatenate([y_train_original, y_new])

print(f"Eski train: {X_train_original.shape}")
print(f"Yeni train: {X_train_combined.shape}")

# Eğit
print("\nModel eğitiliyor...")
rf_new = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_new.fit(X_train_combined, y_train_combined)

# Test
y_pred = rf_new.predict(X_test)
y_prob = rf_new.predict_proba(X_test)[:,1]
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nSonuçlar:")
print(f"Accuracy: {acc:.4f}")
print(f"F1: {f1:.4f}")
print(f"AUC: {auc:.4f}")

Eski train: (24987, 64)
Yeni train: (40017, 64)

Model eğitiliyor...

Sonuçlar:
Accuracy: 0.9329
F1: 0.9330
AUC: 0.9817


In [58]:
test_urls = [
    "google.com",
    "gov.tr",
    "onedio.com",
    "wikipedia.org",
    "http://paypal-login-verify.xyz",
    "http://amazon-secure-update.tk",
    "https://google.com",
    "https://gov.tr",
]

print("Test (protokolsüz ve protokollü):")
print("-" * 60)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = rf_new.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Test (protokolsüz ve protokollü):
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → google.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → gov.tr

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → onedio.com

 Feature extraction basliyor...
   [1/4] URL fe

In [59]:
# Önceki modeli geri yükle (5028 URL eklenmiş olan)
# Onu yeniden eğitmemiz lazım

X_train_original = np.load('data/processed/X_train.npy')
y_train_original = np.load('data/processed/y_train.npy')

rf_iyi = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_iyi.fit(X_train_original, y_train_original)

joblib.dump(rf_iyi, 'data/models/baseline_random_forest.pkl')
print("Orijinal model geri yüklendi!")

Orijinal model geri yüklendi!


In [60]:
test_urls = [
    "google.com",
    "https://google.com",
    "http://paypal-login-verify.xyz",
    "http://amazon-secure-update.tk",
]

print("Orijinal model testi:")
print("-" * 60)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = rf_iyi.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Orijinal model testi:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.49) → google.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
🔴 PHİSHİNG (0.54) → https://google.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
🔴 PHİSHİNG (0.97) → http://paypal-login-verify.xyz

 Feature extraction b

In [61]:
import requests
import zipfile
import io
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
import joblib

# Tranco indir
print("İndiriliyor...")
r = requests.get("https://tranco-list.eu/top-1m.csv.zip", timeout=30)
z = zipfile.ZipFile(io.BytesIO(r.content))
tranco = pd.read_csv(z.open(z.namelist()[0]), header=None, names=['rank', 'domain'])
top_sites = tranco[tranco['rank'] <= 5000]['domain'].tolist()

turkish_sites = [
    "google.com.tr", "gov.tr", "onedio.com", "sahibinden.com",
    "hepsiburada.com", "trendyol.com", "akbank.com",
    "turkcell.com.tr", "hurriyet.com.tr", "ntv.com.tr",
]

new_urls = [f"https://{d}" for d in top_sites] + [f"https://{d}" for d in turkish_sites]
new_df = pd.DataFrame({'url': new_urls, 'timestamp': datetime.now(), 'label': 0, 'source': 'tranco'})

print(f"{len(new_urls)} URL hazırlandı, feature extraction başlıyor...")
new_features = pipeline.transform(new_df, verbose=True)

feature_cols = [c for c in new_features.columns if c not in ['url', 'label', 'timestamp', 'source']]
X_new = new_features[feature_cols].reindex(columns=feature_names, fill_value=0).values
y_new = np.zeros(len(X_new))

X_train_original = np.load('data/processed/X_train.npy')
y_train_original = np.load('data/processed/y_train.npy')

X_combined = np.vstack([X_train_original, X_new])
y_combined = np.concatenate([y_train_original, y_new])

print("Model eğitiliyor...")
rf_best = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_best.fit(X_combined, y_combined)

joblib.dump(rf_best, 'data/models/baseline_random_forest.pkl')
print("✅ Model kaydedildi!")

İndiriliyor...
5010 URL hazırlandı, feature extraction başlıyor...

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...


URL Features: 100%|██████████████████████████████████████████████████████████████████████| 5010/5010 [00:00<00:00, 26672.35it/s]


   [2/4] Adversarial features cikariliyor...


Adversarial Features: 100%|████████████████████████████████████████████████████████████████| 5010/5010 [00:11<00:00, 423.10it/s]


   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...


Cialdini Features: 100%|█████████████████████████████████████████████████████████████████| 5010/5010 [00:00<00:00, 44267.20it/s]


Feature extraction tamamlandi: 5010 satir, 64 feature
Model eğitiliyor...
✅ Model kaydedildi!


In [62]:
test_urls = [
    "https://google.com",
    "https://gov.tr",
    "https://onedio.com",
    "https://sahibinden.com",
    "http://paypal-login-verify.xyz",
    "http://amazon-secure-update.tk",
]

print("Test:")
print("-" * 60)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = rf_best.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Test:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://google.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://gov.tr

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://onedio.com

 Feature extraction basliyor...
   [1/4] URL featur

In [63]:
from sklearn.metrics import accuracy_score

# Train seti performansı
y_train_pred = rf_best.predict(X_combined)
train_acc = accuracy_score(y_combined, y_train_pred)

# Test seti performansı
y_test_pred = rf_best.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Fark: {train_acc - test_acc:.4f}")
print()
if train_acc - test_acc > 0.05:
    print("⚠️ OVERFITTING VAR! Fark %5'ten büyük.")
else:
    print("✅ Overfitting yok. Fark %5'ten küçük.")

Train accuracy: 0.9999
Test accuracy:  0.9348
Fark: 0.0651

⚠️ OVERFITTING VAR! Fark %5'ten büyük.


In [64]:
df = pd.read_csv('data/processed/raw_dataset.csv')
mesru = df[df['label'] == 0]['url']

https_count = mesru.str.startswith('https://').sum()
http_count = mesru.str.startswith('http://').sum()
no_protocol = (~mesru.str.startswith('http')).sum()

print(f"https:// meşru: {https_count}")
print(f"http:// meşru: {http_count}")
print(f"Protokolsüz meşru: {no_protocol}")

https:// meşru: 11634
http:// meşru: 104
Protokolsüz meşru: 13260


In [65]:
phishing = df[df['label'] == 1]['url']

https_ph = phishing.str.startswith('https://').sum()
http_ph = phishing.str.startswith('http://').sum()
no_proto_ph = (~phishing.str.startswith('http')).sum()

print(f"https:// phishing: {https_ph}")
print(f"http:// phishing: {http_ph}")
print(f"Protokolsüz phishing: {no_proto_ph}")

https:// phishing: 900
http:// phishing: 11286
Protokolsüz phishing: 12803
